In [22]:
import pandas as pd, requests, joblib

In [23]:
INDICATORS = {
    'EG.ELC.ACCS.ZS': 'electricity_access',
    'NV.AGR.TOTL.ZS': 'agri_value_added',
    'SH.XPD.CHEX.GD.ZS': 'health_expenditure',
    'NY.GDP.MKTP.KD.ZG': 'gdp_growth',
    'NY.GDP.PCAP.CD': 'gdp_per_capita',
    'SE.XPD.TOTL.GD.ZS': 'education_expenditure',
    'IT.NET.USER.ZS': 'internet_users',
    'FP.CPI.TOTL.ZG': 'inflation',
    'SP.DYN.LE00.IN': 'life_expectancy',
    'SP.URB.TOTL.IN.ZS': 'urban_population',
    'SI.POV.DDAY': 'poverty_ratio',
}

YEARS = '2023:2024'
frames = []

In [24]:
for code, name in INDICATORS.items():
    url = (f'https://api.worldbank.org/v2/country/all/indicator/{code}'
           f'?format=json&date={YEARS}&per_page=20000')
    payload = requests.get(url, timeout=60).json()
    if len(payload) < 2 or payload[1] is None:
        print('no data for', name)
        continue
    rows = [{'country_code': r['countryiso3code'],
             'year': int(r['date']),
             name: r['value']} for r in payload[1] if r['countryiso3code']]
    frames.append(pd.DataFrame(rows).set_index(['country_code', 'year']))
    print(name, 'ok')

electricity_access ok
agri_value_added ok
health_expenditure ok
gdp_growth ok
gdp_per_capita ok
education_expenditure ok
internet_users ok
inflation ok
life_expectancy ok
urban_population ok
poverty_ratio ok


In [25]:
panel = pd.concat(frames, axis=1).reset_index()
panel.shape

(520, 13)

In [26]:
print('rows with a poverty value:', panel['poverty_ratio'].notna().sum())

rows with a poverty value: 102


In [27]:
panel = panel.dropna(subset=['poverty_ratio'])
predictors = [c for c in panel.columns if c not in ['country_code', 'year', 'poverty_ratio']]
panel[predictors] = panel[predictors].fillna(panel[predictors].median())

In [28]:
#  merge region and build the one-hot columns explicitly
meta_json = requests.get(
    'https://api.worldbank.org/v2/country?format=json&per_page=400', timeout=30
).json()[1]
meta = pd.DataFrame([{'country_code': c['id'], 'region': c['region']['value']}
                     for c in meta_json])
meta = meta[meta['region'] != 'Aggregates']

panel = panel.merge(meta, on='country_code', how='left')



In [29]:
# Map region names to the model's exact column names
REGION_MAP = {
    'East Asia & Pacific': None,   # baseline: all zeros
    'Europe & Central Asia': 'reg_europe_and_central_asia',
    'Latin America & Caribbean': 'reg_latin_america_and_caribbean',
    'Middle East, North Africa, Afghanistan & Pakistan': 'reg_middle_east_north_africa_afghanistan_and_pakistan',
    'North America': 'reg_north_america',
    'South Asia': 'reg_south_asia',
    'Sub-Saharan Africa': 'reg_sub_saharan_africa',
}

for col in REGION_MAP.values():
    if col is not None:
        panel[col] = 0

for region_name, col in REGION_MAP.items():
    if col is not None:
        panel.loc[panel['region'] == region_name, col] = 1



In [30]:
print(panel['region'].isnull().sum(), 'rows failed to match a region')
print(panel.loc[panel['region'].isnull(), 'country_code'].unique()[:20])

20 rows failed to match a region
['EAS' 'ECS' 'IDA' 'LCN' 'LMY' 'MEA' 'NAC' 'SAS' 'SSF' 'WLD']


In [31]:
# Aggregate rows (WLD, EAS, SSF, IDA, ...) are not countries
print('dropping', panel['region'].isnull().sum(), 'aggregate rows')
panel = panel.dropna(subset=['region'])

dropping 20 aggregate rows


In [32]:
#
feature_names = joblib.load('../API/feature_names.pkl')
for col in feature_names:
    if col not in panel.columns:
        panel[col] = 0
panel[feature_names + ['poverty_ratio']].to_csv('retrain_new_data.csv', index=False)